# Key Features
#### saves all the listed entities for NYSE, NASDAQ, NSE and BSE to \ljmu\Data\listed_csv

In [1]:
import pandas as pd
from pathlib import Path
from time import sleep
from io import StringIO
import requests

import requests
from pathlib import Path
from time import sleep
from io import StringIO
import pandas as pd

# =========================
# Directories
# =========================
RAW_DIR = Path("../Data/listed_csv")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Index URLs
# =========================
URLS = {
    # US (DataHub)
    "NASDAQ": "https://datahub.io/core/nasdaq-listings/r/nasdaq-listed-symbols.csv",
    "NYSE": "https://datahub.io/core/nyse-other-listings/_r/-/data/nyse-listed.csv",
}

# Browser-like headers (CRITICAL for NIFTY)
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/csv,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Referer": "https://www.niftyindices.com/",
    "Connection": "keep-alive",
}

# =========================
# Download with retries
# =========================
def download_with_retry(session, name, url, retries=3):
    for attempt in range(1, retries + 1):
        try:
            print(f"Downloading {name} , {url}, (attempt {attempt})")
            resp = session.get(url, headers=HEADERS, timeout=120)
            resp.raise_for_status()
            return resp.content
        except Exception as e:
            print(f"⚠️ Attempt {attempt} failed for {name}: {e}")
            sleep(5 * attempt)
    raise RuntimeError(f"❌ Failed after {retries} attempts: {name}")


# =========================
# Download all raw CSVs
# =========================
def download_raw():
    with requests.Session() as session:
        for name, url in URLS.items():
            content = download_with_retry(session, name, url)
            file_path = RAW_DIR / f"{name}.csv"
            file_path.write_bytes(content)
            print(f"✅ Saved raw file → {file_path}")

 


 




In [2]:
import pandas as pd

def download_nse(url,file_path):
    df = pd.read_csv(url)
    # Clean column names
    df.columns = df.columns.str.strip()
    # Keep only active equity stocks
    df = df[df["SERIES"] == "EQ"]
    # Select useful columns
    df = df[["SYMBOL", "NAME OF COMPANY", "ISIN NUMBER"]]
    df.columns = ["Symbol", "Company", "ISIN"]
    # Create unified ticker
    df["Ticker"] = "NSE:" + df["Symbol"]
    df.to_csv(r"C:\ljmu\ljmu\Data\listed_csv\NSE.csv", index=False)
    print(f"NSE rows: {len(df)}")
    print("NSE download from: "+url+"Saved to:", file_path)

    return df

In [3]:
import pandas as pd
import requests
def download_bse(file_path=r"C:\ljmu\ljmu\Data\listed_csv\BSE.csv"):
    url = "https://stocktrendsindia.appspot.com/BSE-Scrip-Code-Name-Sector-Download.csv"
    df_bse = pd.read_csv(url)
    df_bse.to_csv(r"C:\ljmu\ljmu\Data\listed_csv\BSE.csv", index=False)
    print(f"✅ {len(df_bse)} BSE companies downloaded")
    print("NSE download from: "+url+"Saved to:", file_path)


In [4]:
# =========================
# Main
# =========================
def main():
    download_raw()
    download_nse(url = "https://archives.nseindia.com/content/equities/EQUITY_L.csv",file_path=r"C:\ljmu\ljmu\Data\listed_csv\NSE.csv")
    download_bse(file_path=r"C:\ljmu\ljmu\Data\listed_csv\BSE.csv")
if __name__ == "__main__":
    main()

✅ Saved raw file → ..\Data\listed_csv\NASDAQ.csv
✅ Saved raw file → ..\Data\listed_csv\NYSE.csv
NSE rows: 2107
NSE download from: https://archives.nseindia.com/content/equities/EQUITY_L.csvSaved to: C:\ljmu\ljmu\Data\listed_csv\NSE.csv
✅ 7413 BSE companies downloaded
NSE download from: https://stocktrendsindia.appspot.com/BSE-Scrip-Code-Name-Sector-Download.csvSaved to: C:\ljmu\ljmu\Data\listed_csv\BSE.csv
